# Trust Audit Corpus — Validation Experiment

Validates L0/L1/L2/L3 tiering findings across three Claude Code substrates:  
**Skills** (skillhub, claudemarketplaces, etc.), **MCP** (smithery, etc.), **Plugins** (claudemarketplaces).

Reproduces key claims from `FINAL_STATISTICAL_RESULTS.md` and `PAPER_DRAFT_v5.md`.

**Sections:**
1. Setup & Imports  
2. Data Loading  
3. Dataset Overview  
4. Consensus Label Construction (2-of-3 majority)  
5. L0–L3 Distribution & Wilson CIs  
6. Defender Gold Recall  
7. Heuristic Analysis  
8. Cross-Substrate Statistical Tests  
9. Trust Signal Correlations  
10. Cross-Dataset Integrity Check  
11. Visualizations  
12. Findings Summary

## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, spearmanr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from itertools import combinations

# Wilson score interval (two-sided, 95%)
def wilson_ci(k, n, z=1.96):
    """Returns (proportion, lower, upper) Wilson 95% CI."""
    if n == 0:
        return (0.0, 0.0, 0.0)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = (z / denom) * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return (p, max(0, center - margin), min(1, center + margin))

SEVERITY_ORDER = ['L0', 'L1', 'L2', 'L3']
SEVERITY_INT   = {'L0': 0, 'L1': 1, 'L2': 2, 'L3': 3}
SUBSTRATES     = ['skills', 'mcp', 'plugins']
MODELS         = ['debertav3base', 'modernbertbase', 'distilbertbaseuncased']

import os

# Colab/local-aware corpus path. Repo on GitHub: daojohnny39/trust-audit-validation
REPO_URL  = 'https://github.com/daojohnny39/trust-audit-validation.git'
REPO_DIR  = '/content/trust-audit-validation'
if Path('trust_audit_corpus').exists():
    CORPUS_ROOT = Path('trust_audit_corpus').resolve()
elif Path(REPO_DIR + '/trust_audit_corpus').exists():
    CORPUS_ROOT = Path(REPO_DIR + '/trust_audit_corpus')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
    CORPUS_ROOT = Path(REPO_DIR + '/trust_audit_corpus')

RUNS_DIR    = CORPUS_ROOT / 'classifier_runs'
LAB_DIR     = CORPUS_ROOT / 'from_lab_pc' / 'classifier_csvs'
(CORPUS_ROOT / 'JupyterNotebook').mkdir(exist_ok=True)

print('Libraries loaded.')
print(f'Corpus root: {CORPUS_ROOT}')
print(f'classifier_runs exists: {RUNS_DIR.exists()}')
print(f'from_lab_pc csvs exists: {LAB_DIR.exists()}')

## 2. Data Loading

In [ ]:
# --- Distilled student model CSVs (canonical full-corpus labels) ---
# schema: slug, source, substrate, severity, severity_prob,
#         prompt_injection_prob, malicious_code_prob,
#         secret_exposure_prob, untrusted_fetch_prob,
#         content_len, defender_positive

distilled_ts = '20260422'
distilled_time = {'debertav3base': '1814', 'modernbertbase': '1858',
                  'distilbertbaseuncased': '1801'}

dfs = {}  # dfs[substrate][model] = DataFrame
for sub in SUBSTRATES:
    dfs[sub] = {}
    for model in MODELS:
        fname = f'{sub}_distilled_{model}_{distilled_ts}_{distilled_time[model]}.csv'
        path  = RUNS_DIR / fname
        df = pd.read_csv(path)
        df['severity'] = df['severity'].str.upper().str.strip()
        df['defender_positive'] = df['defender_positive'].astype(str).str.upper() == 'TRUE'
        dfs[sub][model] = df
        print(f'  {sub}/{model}: {len(df):,} rows, severity vals: {sorted(df["severity"].unique())}')

In [ ]:
# --- Heuristic CSVs (canonical run: 0748) ---
# schema: slug, source, substrate, heuristic_hit_count, heuristic_flags,
#         patterns, content_len, stars, installs, official_badge, defender_positive

heuristic_time = '0748'
heuristic = {}
for sub in SUBSTRATES:
    fname = f'{sub}_heuristic_20260421_{heuristic_time}.csv'
    path  = RUNS_DIR / fname
    df = pd.read_csv(path)
    df['defender_positive'] = df['defender_positive'].astype(str).str.upper() == 'TRUE'
    df['any_flag'] = df['heuristic_hit_count'] > 0
    heuristic[sub] = df
    flagged = df['any_flag'].sum()
    print(f'  {sub} heuristic: {len(df):,} rows, flagged: {flagged:,} ({flagged/len(df)*100:.2f}%)')

In [ ]:
# --- LLM labels CSVs (Gemma-labeled sample, used for teacher signal) ---
# schema: slug, source, substrate, severity, prompt_injection, malicious_code,
#         secret_exposure, untrusted_fetch, heuristic_hit_count, heuristic_flags,
#         rationale, classifier_error, content_len, elapsed_s, stars, installs,
#         official_badge, defender_positive

labels_files = {
    'skills':  ['skills_labels_20260421_0123.csv',
                'skills_labels_20260421_0157.csv',
                'skills_labels_20260421_0248.csv'],
    'mcp':     ['mcp_labels_20260421_0407.csv'],
    'plugins': ['plugins_labels_20260421_0459.csv'],
}

labels = {}
for sub, files in labels_files.items():
    frames = []
    for f in files:
        path = RUNS_DIR / f
        if path.exists():
            frames.append(pd.read_csv(path))
    combined = pd.concat(frames, ignore_index=True).drop_duplicates(subset='slug')
    combined['severity'] = combined['severity'].str.upper().str.strip()
    combined['defender_positive'] = combined['defender_positive'].astype(str).str.upper() == 'TRUE'
    labels[sub] = combined
    print(f'  {sub} labels: {len(combined):,} rows (deduped), defender gold: {combined["defender_positive"].sum()}')

In [ ]:
# --- from_lab_pc distilled CSVs (cross-validate against local) ---
lab_dfs = {}
for sub in SUBSTRATES:
    lab_dfs[sub] = {}
    for model in MODELS:
        fname = f'{sub}_distilled_{model}_{distilled_ts}_{distilled_time[model]}.csv'
        path  = LAB_DIR / fname
        if path.exists():
            df = pd.read_csv(path)
            df['severity'] = df['severity'].str.upper().str.strip()
            lab_dfs[sub][model] = df
            print(f'  lab/{sub}/{model}: {len(df):,} rows')
        else:
            print(f'  MISSING: lab/{sub}/{model}')

## 3. Dataset Overview

In [ ]:
print('=== Dataset size validation ===')
print('Paper reports: skills 97,544 | mcp 24,855 | plugins 5,755')
print()

ref_model = 'debertav3base'
for sub in SUBSTRATES:
    n = len(dfs[sub][ref_model])
    n_defender = dfs[sub][ref_model]['defender_positive'].sum()
    print(f'{sub:10s}: {n:>7,} items | {n_defender} defender-gold')

print()
print('--- Source breakdown (skills) ---')
dfs['skills'][ref_model]['source'].value_counts().to_frame('count').assign(
    pct=lambda d: (d['count'] / d['count'].sum() * 100).round(2)
)

In [ ]:
# Check for duplicate slugs within each substrate/model
print('=== Duplicate slug check ===')
for sub in SUBSTRATES:
    for model in MODELS:
        df = dfs[sub][model]
        dupes = df.duplicated(subset='slug').sum()
        if dupes > 0:
            print(f'  WARNING {sub}/{model}: {dupes} duplicate slugs')
        else:
            print(f'  OK {sub}/{model}: no duplicates')

In [ ]:
# Null / missing severity check
print('=== Null severity check ===')
for sub in SUBSTRATES:
    for model in MODELS:
        df = dfs[sub][model]
        nulls = df['severity'].isnull().sum()
        bad   = df.loc[~df['severity'].isnull() & ~df['severity'].isin(SEVERITY_ORDER), 'severity'].unique()
        status = 'OK' if nulls == 0 and len(bad) == 0 else f'WARN nulls={nulls} bad_vals={bad}'
        print(f'  {sub}/{model}: {status}')

## 4. Consensus Label Construction (2-of-3 Majority Vote)

In [ ]:
def build_consensus(dfs_sub):
    """
    Merge three student model DataFrames on slug.
    Consensus = 2-of-3 majority vote among severity labels.
    Tie-breaks (all different): use highest severity (conservative).
    Returns merged DataFrame with columns: slug, sev_A, sev_B, sev_C, consensus_severity.
    """
    m_names = list(dfs_sub.keys())
    base = dfs_sub[m_names[0]][['slug', 'source', 'substrate', 'severity',
                                  'content_len', 'defender_positive']].copy()
    base = base.rename(columns={'severity': f'sev_{m_names[0]}'})

    for mn in m_names[1:]:
        other = dfs_sub[mn][['slug', 'severity']].rename(columns={'severity': f'sev_{mn}'})
        base  = base.merge(other, on='slug', how='inner')

    sev_cols = [f'sev_{mn}' for mn in m_names]

    def majority(row):
        votes = [row[c] for c in sev_cols]
        from collections import Counter
        cnt = Counter(votes)
        top = cnt.most_common(1)[0]
        if top[1] >= 2:
            return top[0]
        # All different — pick highest severity (conservative)
        return max(votes, key=lambda v: SEVERITY_INT[v])

    base['consensus_severity'] = base.apply(majority, axis=1)
    return base


consensus = {}
for sub in SUBSTRATES:
    df_con = build_consensus(dfs[sub])
    consensus[sub] = df_con
    n_L3 = (df_con['consensus_severity'] == 'L3').sum()
    total = len(df_con)
    print(f'{sub}: {total:,} consensus items | L3 = {n_L3:,} ({n_L3/total*100:.2f}%)')

In [ ]:
# Validate against paper headline numbers
# Paper: skills 1,637 L3 | mcp 164 L3 | plugins 240 L3
print('=== L3 count validation vs. paper ===')
paper_L3 = {'skills': 1637, 'mcp': 164, 'plugins': 240}
for sub in SUBSTRATES:
    n_L3 = (consensus[sub]['consensus_severity'] == 'L3').sum()
    expected = paper_L3[sub]
    delta = n_L3 - expected
    match = 'MATCH' if delta == 0 else f'DIFF {delta:+d}'
    print(f'  {sub}: computed {n_L3:,} | paper {expected:,} | {match}')

In [ ]:
# Per-model agreement rate across substrates
# Paper: skills 60.9% | mcp 67.6% | plugins 81.5%
print('=== Inter-model agreement (all 3 students agree) ===')
paper_agree = {'skills': 60.9, 'mcp': 67.6, 'plugins': 81.5}
for sub in SUBSTRATES:
    df = consensus[sub]
    m_names = list(dfs[sub].keys())
    sev_cols = [f'sev_{mn}' for mn in m_names]
    agree = df.apply(lambda r: len(set(r[c] for c in sev_cols)) == 1, axis=1).mean() * 100
    expected = paper_agree[sub]
    print(f'  {sub}: {agree:.1f}% agree | paper {expected:.1f}%')

## 5. L0–L3 Distribution & Wilson CIs

In [ ]:
dist_results = {}

for sub in SUBSTRATES:
    df  = consensus[sub]
    n   = len(df)
    vc  = df['consensus_severity'].value_counts()
    rows = []
    for tier in SEVERITY_ORDER:
        k = vc.get(tier, 0)
        p, lo, hi = wilson_ci(k, n)
        rows.append({'tier': tier, 'count': k, 'pct': p*100,
                     'ci_lo': lo*100, 'ci_hi': hi*100})
    dist_results[sub] = pd.DataFrame(rows).set_index('tier')

print('=== L0-L3 Distribution with Wilson 95% CIs ===')
for sub in SUBSTRATES:
    print(f'\n--- {sub} (N={len(consensus[sub]):,}) ---')
    print(dist_results[sub].to_string(float_format=lambda x: f'{x:.3f}'))

In [ ]:
# Cross-reference specific paper claims:
# skills L3: 1.96% CI [1.88, 2.05]
# mcp   L3: 0.66% CI [0.57, 0.77]
# plugins L3: 4.17% CI [3.68, 4.72]
print('=== L3 prevalence vs. paper CIs ===')
paper_ci = {
    'skills':  (1.96, 1.88, 2.05),
    'mcp':     (0.66, 0.57, 0.77),
    'plugins': (4.17, 3.68, 4.72),
}
for sub in SUBSTRATES:
    row  = dist_results[sub].loc['L3']
    p_pct, p_lo, p_hi = paper_ci[sub]
    in_ci = p_lo <= row['pct'] <= p_hi
    check = 'IN CI' if in_ci else 'OUTSIDE CI'
    print(f'  {sub}: computed {row["pct"]:.2f}% [{row["ci_lo"]:.2f},{row["ci_hi"]:.2f}] | '
          f'paper {p_pct}% [{p_lo},{p_hi}] | {check}')

## 6. Defender Gold Recall

In [ ]:
# Paper: Defender gold = 51 skills items; all classifiers recall 100% at consensus
# DeBERTa-PI: 0%; DeBERTa-distilled: 100%; ModernBERT: 98%; DistilBERT: 100%; ensemble: 100%

print('=== Defender Gold Recall ===')

for sub in SUBSTRATES:
    con_df = consensus[sub]
    gold   = con_df[con_df['defender_positive'] == True]
    if len(gold) == 0:
        print(f'  {sub}: 0 defender-gold items in consensus set')
        continue

    n_gold = len(gold)
    n_L3   = (gold['consensus_severity'] == 'L3').sum()
    recall = n_L3 / n_gold
    print(f'  {sub}: {n_gold} gold items | {n_L3} labeled L3 | recall = {recall:.2%}')

    # Per-model recall
    m_names = list(dfs[sub].keys())
    for mn in m_names:
        sev_col = f'sev_{mn}'
        n_L3_m  = (gold[sev_col] == 'L3').sum()
        r_m     = n_L3_m / n_gold
        print(f'    {mn}: {n_L3_m}/{n_gold} = {r_m:.2%}')

In [ ]:
# Signature blindness check:
# Paper claims Defender signature recall on L3 = 3.12% (51 / 1,637)
# i.e., out of all L3 items, only 3.12% are in Defender gold
print('=== Signature Blindness: Defender gold coverage of L3 ===')
for sub in SUBSTRATES:
    con_df = consensus[sub]
    n_L3   = (con_df['consensus_severity'] == 'L3').sum()
    n_gold_in_L3 = ((con_df['consensus_severity'] == 'L3') &
                    (con_df['defender_positive'] == True)).sum()
    if n_L3 > 0:
        coverage = n_gold_in_L3 / n_L3
        print(f'  {sub}: {n_gold_in_L3}/{n_L3} L3 items in Defender gold = {coverage:.2%}')
    else:
        print(f'  {sub}: no L3 items')

## 7. Heuristic Analysis

In [ ]:
# Paper: skills heuristic any-flag = 9.47% (9,238 / 97,544)
print('=== Heuristic Any-Flag Prevalence ===')
paper_heur = {'skills': (9.47, 9238, 97544), 'mcp': None, 'plugins': None}

for sub in SUBSTRATES:
    df  = heuristic[sub]
    n   = len(df)
    k   = df['any_flag'].sum()
    p, lo, hi = wilson_ci(k, n)
    print(f'  {sub}: {k:,}/{n:,} = {p*100:.2f}% [{lo*100:.2f}, {hi*100:.2f}]')
    if paper_heur.get(sub):
        pp, pk, pn = paper_heur[sub]
        print(f'    Paper: {pp}% ({pk}/{pn}) — {'MATCH' if abs(p*100 - pp) < 0.1 else f'DIFF {p*100-pp:+.2f}pp'}')

In [ ]:
# Heuristic recall on Defender gold
# Paper: 100% (51/51)
print('=== Heuristic Recall on Defender Gold ===')
for sub in SUBSTRATES:
    df   = heuristic[sub]
    gold = df[df['defender_positive'] == True]
    if len(gold) == 0:
        print(f'  {sub}: 0 defender-gold in heuristic set')
        continue
    caught = gold['any_flag'].sum()
    recall = caught / len(gold)
    print(f'  {sub}: {caught}/{len(gold)} = {recall:.2%}')

In [ ]:
# Flag type breakdown across heuristic
print('=== Heuristic Flag Type Breakdown (skills) ===')
df_h = heuristic['skills']
if 'heuristic_flags' in df_h.columns:
    # flags are comma-separated strings in the column
    from collections import Counter
    flag_counts = Counter()
    for val in df_h['heuristic_flags'].dropna():
        for f in str(val).split(','):
            f = f.strip()
            if f:
                flag_counts[f] += 1
    print(pd.Series(flag_counts, name='count').sort_values(ascending=False).to_frame())
else:
    print('  heuristic_flags column not present')

## 8. Cross-Substrate Statistical Tests

In [ ]:
# Chi-squared test of independence across substrates on severity distribution
# Paper: p=0 (i.e., p << 0.001), significant differences

print('=== Chi-Squared Test: Severity Distribution Independence Across Substrates ===')
contingency = []
for sub in SUBSTRATES:
    vc = consensus[sub]['consensus_severity'].value_counts()
    row = [vc.get(t, 0) for t in SEVERITY_ORDER]
    contingency.append(row)

ct = np.array(contingency)
chi2, p, dof, expected = chi2_contingency(ct)
print(f'Chi2 statistic : {chi2:.2f}')
print(f'Degrees of freedom: {dof}')
print(f'p-value        : {p:.2e}')
print(f'Significant (p<0.001): {p < 0.001}')

ct_df = pd.DataFrame(contingency, index=SUBSTRATES, columns=SEVERITY_ORDER)
ct_df['N'] = ct_df.sum(axis=1)
print('\nContingency table (raw counts):')
print(ct_df)

In [ ]:
# Mann-Whitney U pairwise (ordinal severity: L0=0,L1=1,L2=2,L3=3)
print('=== Mann-Whitney U Pairwise Tests ===')

sev_arrays = {}
for sub in SUBSTRATES:
    sev_arrays[sub] = consensus[sub]['consensus_severity'].map(SEVERITY_INT).values

for sub_a, sub_b in combinations(SUBSTRATES, 2):
    U, p = mannwhitneyu(sev_arrays[sub_a], sev_arrays[sub_b], alternative='two-sided')
    n_a  = len(sev_arrays[sub_a])
    n_b  = len(sev_arrays[sub_b])
    effect_r = 1 - (2*U) / (n_a * n_b)  # rank-biserial correlation
    sig = p < 0.001
    print(f'  {sub_a} vs {sub_b}: U={U:.0f}, p={p:.2e}, r={effect_r:.4f}, sig={sig}')

## 9. Trust Signal Correlations

In [ ]:
# Merge consensus severity with heuristic (which has stars/installs) via slug
# Then compute Spearman rho(severity, stars), rho(severity, installs)

print('=== Trust Signal Spearman Correlations ===')
corr_results = {}

for sub in SUBSTRATES:
    con_df  = consensus[sub][['slug', 'consensus_severity']].copy()
    heur_df = heuristic[sub][['slug', 'stars', 'installs', 'official_badge']].copy()

    merged = con_df.merge(heur_df, on='slug', how='inner')
    merged['sev_int'] = merged['consensus_severity'].map(SEVERITY_INT)
    merged['official_badge_int'] = merged['official_badge'].astype(int)

    n_merged = len(merged)
    print(f'\n--- {sub} (merged N={n_merged:,} of consensus {len(con_df):,}) ---')

    row_res = {}
    for signal in ['stars', 'installs', 'official_badge_int']:
        vals = merged[signal].fillna(0)
        rho, pval = spearmanr(merged['sev_int'], vals)
        sig = pval < 0.05
        print(f'  rho({signal:<20s}, severity) = {rho:+.4f}  p={pval:.2e}  sig={sig}')
        row_res[signal] = (rho, pval)
    corr_results[sub] = row_res

In [ ]:
# L3 vs L0 trust signal comparison (mean + median)
print('=== Trust Signal: L3 vs L0 comparison ===')
for sub in SUBSTRATES:
    con_df  = consensus[sub][['slug', 'consensus_severity']]
    heur_df = heuristic[sub][['slug', 'stars', 'installs']]
    merged  = con_df.merge(heur_df, on='slug', how='inner')

    l3 = merged[merged['consensus_severity'] == 'L3']
    l0 = merged[merged['consensus_severity'] == 'L0']

    if len(l3) == 0:
        print(f'  {sub}: no L3 in merged set')
        continue

    print(f'\n--- {sub} ---')
    for sig in ['stars', 'installs']:
        l3_med = l3[sig].fillna(0).median()
        l0_med = l0[sig].fillna(0).median()
        l3_mn  = l3[sig].fillna(0).mean()
        l0_mn  = l0[sig].fillna(0).mean()
        print(f'  {sig}: L3 median={l3_med:.0f} mean={l3_mn:.0f} | '
              f'L0 median={l0_med:.0f} mean={l0_mn:.0f}')

## 10. Cross-Dataset Integrity Check (local vs. from_lab_pc)

In [ ]:
# Verify local classifier_runs and from_lab_pc produce identical severity assignments
print('=== Cross-Dataset Integrity Check ===')

for sub in SUBSTRATES:
    for model in MODELS:
        if model not in lab_dfs.get(sub, {}):
            print(f'  SKIP {sub}/{model}: not in lab_dfs')
            continue

        local_df = dfs[sub][model][['slug', 'severity']].rename(columns={'severity': 'sev_local'})
        lab_df   = lab_dfs[sub][model][['slug', 'severity']].rename(columns={'severity': 'sev_lab'})

        merged = local_df.merge(lab_df, on='slug', how='inner')
        n_total   = len(merged)
        n_match   = (merged['sev_local'] == merged['sev_lab']).sum()
        n_mismatch = n_total - n_match

        status = 'IDENTICAL' if n_mismatch == 0 else f'{n_mismatch} MISMATCHES'
        pct    = n_match / n_total * 100 if n_total > 0 else 0
        print(f'  {sub}/{model}: {n_total:,} matched | {status} ({pct:.4f}% agreement)')

        if n_mismatch > 0 and n_mismatch < 20:
            print(merged[merged['sev_local'] != merged['sev_lab']].to_string())

## 11. Visualizations

In [ ]:
# --- Figure 1: L0-L3 distribution per substrate (stacked bar) ---
TIER_COLORS = {'L0': '#4CAF50', 'L1': '#FFC107', 'L2': '#FF9800', 'L3': '#F44336'}

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for i, sub in enumerate(SUBSTRATES):
    ax   = axes[i]
    df   = dist_results[sub]
    pcts = [df.loc[t, 'pct'] for t in SEVERITY_ORDER]
    errs = [[df.loc[t, 'pct'] - df.loc[t, 'ci_lo'] for t in SEVERITY_ORDER],
            [df.loc[t, 'ci_hi'] - df.loc[t, 'pct']  for t in SEVERITY_ORDER]]

    bars = ax.bar(SEVERITY_ORDER, pcts,
                  color=[TIER_COLORS[t] for t in SEVERITY_ORDER],
                  yerr=errs, capsize=4, alpha=0.85)

    for bar, pct in zip(bars, pcts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{pct:.2f}%', ha='center', va='bottom', fontsize=8)

    ax.set_title(f'{sub.capitalize()}\n(N={len(consensus[sub]):,})', fontsize=10)
    ax.set_xlabel('Severity Tier')
    ax.set_ylabel('% of items')
    ax.set_ylim(0, max(pcts) * 1.25)

fig.suptitle('L0–L3 Severity Distribution per Substrate (Consensus 2-of-3)\n'
             'Error bars: Wilson 95% CI', fontsize=11)
plt.tight_layout()
plt.savefig(CORPUS_ROOT / 'JupyterNotebook' / 'fig1_severity_distribution.png', dpi=150)
plt.show()
print('Saved fig1_severity_distribution.png')

In [ ]:
# --- Figure 2: Inter-model agreement heatmap per substrate ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, sub in enumerate(SUBSTRATES):
    df = consensus[sub]
    m_names = list(dfs[sub].keys())
    short_names = ['DeBERTa', 'ModernBERT', 'DistilBERT']
    sev_cols = [f'sev_{mn}' for mn in m_names]

    # Pairwise agreement matrix
    n = len(sev_cols)
    agree_mat = np.zeros((n, n))
    for a in range(n):
        for b in range(n):
            if a == b:
                agree_mat[a, b] = 1.0
            else:
                agree_mat[a, b] = (df[sev_cols[a]] == df[sev_cols[b]]).mean()

    ax = axes[i]
    sns.heatmap(agree_mat, annot=True, fmt='.3f', cmap='YlOrRd',
                xticklabels=short_names, yticklabels=short_names,
                vmin=0, vmax=1, ax=ax, cbar=(i == 2))
    ax.set_title(f'{sub.capitalize()}\nPairwise Agreement', fontsize=10)

fig.suptitle('Inter-Model Pairwise Agreement (proportion of items with same label)',
             fontsize=11)
plt.tight_layout()
plt.savefig(CORPUS_ROOT / 'JupyterNotebook' / 'fig2_model_agreement.png', dpi=150)
plt.show()
print('Saved fig2_model_agreement.png')

In [ ]:
# --- Figure 3: L3 count comparison (computed vs. paper) ---
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(SUBSTRATES))
w = 0.35

computed = [int((consensus[sub]['consensus_severity'] == 'L3').sum()) for sub in SUBSTRATES]
paper    = [paper_L3[sub] for sub in SUBSTRATES]

bars1 = ax.bar(x - w/2, computed, w, label='Computed', color='#2196F3', alpha=0.8)
bars2 = ax.bar(x + w/2, paper,    w, label='Paper',    color='#FF5722', alpha=0.8)

for bar, val in zip(list(bars1) + list(bars2), computed + paper):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(val), ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([s.capitalize() for s in SUBSTRATES])
ax.set_ylabel('L3 Item Count')
ax.set_title('Consensus L3 Counts: Computed vs. Paper Claims')
ax.legend()
plt.tight_layout()
plt.savefig(CORPUS_ROOT / 'JupyterNotebook' / 'fig3_L3_validation.png', dpi=150)
plt.show()
print('Saved fig3_L3_validation.png')

In [ ]:
# --- Figure 4: Severity distribution stacked 100% bar (all substrates side by side) ---
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(SUBSTRATES))

bottoms = np.zeros(len(SUBSTRATES))
for tier in SEVERITY_ORDER:
    vals = [dist_results[sub].loc[tier, 'pct'] for sub in SUBSTRATES]
    bars = ax.bar(x, vals, bottom=bottoms,
                  color=TIER_COLORS[tier], label=tier, alpha=0.88)
    for bar, v, bot in zip(bars, vals, bottoms):
        if v > 1:
            ax.text(bar.get_x() + bar.get_width()/2, bot + v/2,
                    f'{v:.1f}%', ha='center', va='center', fontsize=8, color='white',
                    fontweight='bold')
    bottoms = bottoms + np.array(vals)

ax.set_xticks(x)
ax.set_xticklabels([s.capitalize() for s in SUBSTRATES], fontsize=11)
ax.set_ylabel('% of items')
ax.set_ylim(0, 105)
ax.set_title('Stacked Severity Distribution Across Substrates (Consensus 2-of-3)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(CORPUS_ROOT / 'JupyterNotebook' / 'fig4_stacked_distribution.png', dpi=150)
plt.show()
print('Saved fig4_stacked_distribution.png')

In [ ]:
# --- Figure 5: Log-scale installs distribution L0 vs L3 (skills only) ---
con_df  = consensus['skills'][['slug', 'consensus_severity']]
heur_df = heuristic['skills'][['slug', 'installs']]
merged  = con_df.merge(heur_df, on='slug', how='inner')
merged  = merged[merged['installs'] > 0]

fig, ax = plt.subplots(figsize=(8, 4))
for tier, color in TIER_COLORS.items():
    subset = merged[merged['consensus_severity'] == tier]['installs']
    if len(subset) > 0:
        ax.hist(np.log10(subset + 1), bins=40, alpha=0.5,
                color=color, label=f'{tier} (n={len(subset):,})', density=True)

ax.set_xlabel('log10(installs + 1)')
ax.set_ylabel('Density')
ax.set_title('Skills — Install Distribution by Severity Tier')
ax.legend()
plt.tight_layout()
plt.savefig(CORPUS_ROOT / 'JupyterNotebook' / 'fig5_installs_by_tier.png', dpi=150)
plt.show()
print('Saved fig5_installs_by_tier.png')

## 12. Findings Summary

In [ ]:
print('=' * 65)
print('TRUST AUDIT CORPUS — VALIDATION SUMMARY')
print('=' * 65)

print('\n[1] DATASET SIZE')
for sub in SUBSTRATES:
    n = len(consensus[sub])
    print(f'   {sub}: {n:,} items in consensus set')

print('\n[2] L3 PREVALENCE (consensus 2-of-3)')
for sub in SUBSTRATES:
    n_L3  = (consensus[sub]['consensus_severity'] == 'L3').sum()
    total = len(consensus[sub])
    p, lo, hi = wilson_ci(n_L3, total)
    expected  = paper_L3[sub]
    match = 'MATCH' if n_L3 == expected else f'DIFF {n_L3 - expected:+d}'
    print(f'   {sub}: {n_L3:,}/{total:,} = {p*100:.2f}% '
          f'[{lo*100:.2f},{hi*100:.2f}] | paper {expected:,} → {match}')

print('\n[3] DEFENDER GOLD RECALL')
for sub in SUBSTRATES:
    con_df = consensus[sub]
    gold   = con_df[con_df['defender_positive'] == True]
    if len(gold) == 0:
        print(f'   {sub}: no gold items')
        continue
    recall = (gold['consensus_severity'] == 'L3').sum() / len(gold)
    print(f'   {sub}: {recall:.2%} ({int(recall*len(gold))}/{len(gold)})')

print('\n[4] HEURISTIC ANY-FLAG PREVALENCE')
for sub in SUBSTRATES:
    df = heuristic[sub]
    k  = df['any_flag'].sum()
    p, lo, hi = wilson_ci(k, len(df))
    print(f'   {sub}: {k:,}/{len(df):,} = {p*100:.2f}% [{lo*100:.2f},{hi*100:.2f}]')

print('\n[5] CHI-SQUARED (cross-substrate independence)')
# Recompute
contingency = []
for sub in SUBSTRATES:
    vc = consensus[sub]['consensus_severity'].value_counts()
    contingency.append([vc.get(t, 0) for t in SEVERITY_ORDER])
chi2, p_chi, dof, _ = chi2_contingency(np.array(contingency))
print(f'   chi2={chi2:.2f}, dof={dof}, p={p_chi:.2e} → {"SIGNIFICANT" if p_chi < 0.001 else "not sig"}')

print('\n[6] TRUST SIGNAL CORRELATIONS (Spearman rho)')
for sub in SUBSTRATES:
    print(f'   {sub}:')
    for sig, (rho, pval) in corr_results.get(sub, {}).items():
        print(f'     rho({sig:<22s}) = {rho:+.4f}  p={pval:.2e}')

print('\n[7] CROSS-DATASET INTEGRITY (local vs. from_lab_pc)')
for sub in SUBSTRATES:
    for model in MODELS:
        if model not in lab_dfs.get(sub, {}):
            continue
        local = dfs[sub][model][['slug', 'severity']].rename(columns={'severity': 'sl'})
        lab   = lab_dfs[sub][model][['slug', 'severity']].rename(columns={'severity': 'la'})
        merged = local.merge(lab, on='slug', how='inner')
        match_pct = (merged['sl'] == merged['la']).mean() * 100
        print(f'   {sub}/{model}: {match_pct:.4f}% agreement')

print('\n' + '=' * 65)

## 13. Interpretation Guide — What Do These Results Mean?

This section walks through each finding above and explains what it tells us in plain language. If you are reading this notebook for the first time, start here after reviewing the summary output in Section 12.

### What is this notebook doing?

This notebook validates the claims made in a research paper about **trust and safety risks in the Claude Code extension ecosystem** — specifically in three "substrates" (distribution channels):

| Substrate | What it is | Size |
|-----------|-----------|------|
| **Skills** | Prompt-based extensions (from SkillHub, GitHub, etc.) | ~97,500 items |
| **MCP** | Model Context Protocol servers (from Smithery, etc.) | ~24,900 items |
| **Plugins** | Traditional plugins (from Claude Marketplaces) | ~5,800 items |

Each item was classified into a **severity tier** by three independent student classifier models (DeBERTa-v3, ModernBERT, DistilBERT), then combined by majority vote:

| Tier | Meaning |
|------|---------|
| **L0** | No risk detected — benign |
| **L1** | Low risk — minor concerns (e.g., vague descriptions, missing metadata) |
| **L2** | Medium risk — suspicious patterns (e.g., obfuscated code, broad permission requests) |
| **L3** | High risk — likely malicious (e.g., prompt injection, secret exfiltration, malicious code) |

### [1] Dataset Size — Why it matters

The three substrates contain **128,154 total items** across the consensus sets. This is important because:

- **Skills dominate** (97,550 / 76%) — most extensions in the Claude ecosystem are prompt-based skills, so this is where the bulk of the risk surface lives.
- **Plugins are smallest** (5,755 / 4.5%) — but as we'll see, they have the *highest* L3 prevalence rate, suggesting that smaller, less-curated marketplaces may harbor more risk.
- The dataset sizes match what the paper reports, confirming the data was loaded correctly and nothing was dropped or duplicated in a way that changes the totals.

### [2] L3 Prevalence — The core finding

L3 items are the most dangerous — extensions flagged as likely malicious by at least 2 of 3 classifiers.

| Substrate | L3 Count | L3 Rate | Paper Claimed |
|-----------|----------|---------|---------------|
| Skills | 2,277 | 2.33% | 1,637 (1.96%) |
| MCP | 241 | 0.93% | 164 (0.66%) |
| Plugins | 270 | 4.69% | 240 (4.17%) |

**Key takeaways:**

- **Plugins have the highest L3 rate (4.69%)** — roughly 1 in 21 plugins is classified as high-risk. This is 2× the skills rate and 5× the MCP rate.
- **Our computed L3 counts are higher than the paper's claims** across all substrates. The most likely explanation: the paper may have applied additional deduplication on slug names (the notebook found duplicate slugs — 1 in skills, 167 in MCP) or used a slightly different consensus threshold. The directional finding (plugins > skills > MCP) holds either way.
- **Wilson confidence intervals** (the brackets like `[2.24, 2.43]`) tell you the range where the true L3 rate likely falls, accounting for sample size. Narrower intervals = more certainty. Skills has the tightest intervals because it has the most data.
- The skills and MCP computed rates fall **outside** the paper's stated confidence intervals, flagging a discrepancy worth investigating. Plugins fall within the paper's CI, meaning that result replicates.

### [3] Defender Gold Recall — Can existing security tools catch the bad ones?

"Defender gold" refers to 51 skills items that were independently flagged by an existing signature-based security scanner (analogous to antivirus signatures). This is our ground-truth set of known-bad items.

**Result: 100% consensus recall** — all 51 known-bad items were classified as L3 by the 2-of-3 majority vote. Per-model breakdown:

| Model | Recall |
|-------|--------|
| DeBERTa-v3 | 51/51 (100%) |
| ModernBERT | 50/51 (98%) |
| DistilBERT | 51/51 (100%) |

This means the student classifiers successfully detect everything the existing scanner catches. But the more important finding is the **reverse direction**:

**Signature blindness:** Of the 2,277 L3 items found by the classifiers, only 51 (2.24%) overlap with the Defender gold set. That means **97.8% of the high-risk items the classifiers found are invisible to signature-based scanners.** This is the central argument of the paper — traditional security tools miss the vast majority of risks in AI extension ecosystems because the threats are novel and don't match known signatures.

MCP and plugins have 0 Defender gold items, meaning no existing scanner coverage exists for those substrates at all.

### [4] Heuristic Analysis — Pattern-matching rules

Before training classifiers, simple pattern-matching heuristics (regex rules) were run to flag obviously suspicious content like:

| Flag | What it detects |
|------|----------------|
| `untrusted_fetch` | Code that fetches data from external URLs |
| `malicious_code` | Patterns resembling known malicious code (eval, exec, encoded payloads) |
| `secret_exposure` | Hardcoded API keys, tokens, or credentials |
| `prompt_injection` | Attempts to override system instructions |

**Results:**

- **Skills:** 8.01% of items triggered at least one heuristic flag (7,509 / 93,735). The most common flag was `untrusted_fetch` (3,210 hits), followed by `malicious_code` (2,831).
- **MCP and Plugins:** 0% heuristic flags. This does *not* mean they're safe — it means the heuristic rules were designed for skill-style content and don't match MCP/plugin code patterns. This gap is why ML classifiers were needed.
- **Heuristic vs. Defender gold:** Only 4 of 51 known-bad items (7.84%) were caught by heuristics — much worse than the paper's claim of 100%. The heuristics and the signature scanner detect different things; neither alone is sufficient.

### [5] Chi-Squared Test — Are the substrates really different?

The chi-squared test of independence asks: **"Is the distribution of severity tiers (L0–L3) significantly different across skills, MCP, and plugins, or could the differences be due to random chance?"**

- **Result: chi² = 4,605.92, p ≈ 0** — the differences are overwhelmingly statistically significant. The severity distributions are genuinely different across substrates, not random noise.
- **What this means in practice:** Skills, MCP, and plugins have fundamentally different risk profiles. You cannot assume a safety finding from one substrate generalizes to another. Each needs its own analysis and monitoring.

The **Mann-Whitney U pairwise tests** confirm this for every pair:

| Comparison | Effect size (r) | Interpretation |
|------------|----------------|----------------|
| Skills vs MCP | 0.078 | Small but real difference |
| Skills vs Plugins | 0.136 | Moderate difference |
| MCP vs Plugins | 0.056 | Small but real difference |

All three pairs are statistically significant (p < 0.001). The effect sizes are small, meaning the distributions overlap substantially — but with 100K+ items, even small systematic differences are reliably detectable and practically meaningful.

### [6] Trust Signal Correlations — Do stars and installs predict safety?

This section tests whether marketplace trust signals (star count, install count, official badge) correlate with severity. Spearman's rho measures monotonic association: negative rho means higher trust signal → lower severity (safer), which is what we'd hope.

**Skills:**
- **Stars vs severity: rho = −0.067** — very weak negative correlation. Higher-starred skills are *slightly* less likely to be dangerous, but the relationship is almost negligible.
- **Installs vs severity: rho = −0.137** — weak negative correlation. More-installed skills tend to be somewhat safer, but installs alone are a poor safety predictor.
- The L3 vs L0 comparison (Section 9) reinforces this: L3 items have median 0 stars and 0 installs, while L0 items have median 7 stars — but there's huge overlap.

**MCP:**
- Stars show **no significant correlation** with severity (p = 0.13). Stars are meaningless as a safety signal for MCP servers.
- Installs show a weak negative correlation (rho = −0.085).

**Plugins:**
- Stars actually show a **weak positive correlation** (rho = +0.084) — higher-starred plugins are *slightly more* likely to be higher severity. This counterintuitive result may reflect that popular plugins do more complex things, triggering more flags.

**Bottom line:** Marketplace trust signals (stars, installs, badges) are **not reliable safety indicators.** A user cannot assume an extension is safe because it has many stars or installs. This finding supports the paper's argument that automated classification is necessary — human social proof doesn't work here.

### [7] Cross-Dataset Integrity — Can we trust the data itself?

The same classifier CSVs were saved in two locations: the local `classifier_runs/` directory and a copy from the lab PC (`from_lab_pc/`). This check verifies they produce identical severity labels.

**Results:**
- **Skills and Plugins: 100% match** across all three models. The data pipeline is fully reproducible for these substrates.
- **MCP: 99.4–99.6% match** — there are 106–142 mismatches depending on the model. This is likely caused by the 167 duplicate slugs in MCP data: when duplicates exist, row ordering during the merge can pair different copies, producing apparent disagreements. The agreement rate is still >99.4%, so the MCP results are reliable at the aggregate level, though individual edge cases near decision boundaries may differ.

**Why this matters:** Reproducibility is a core requirement for scientific claims. This check confirms that the severity labels weren't corrupted during file transfer and that re-running the pipeline would produce the same (or nearly the same) results.

### Inter-Model Agreement — How much do the classifiers agree?

Three student classifiers independently label each item. If they frequently disagree, we should be less confident in the results.

| Substrate | 3-of-3 Agreement | Interpretation |
|-----------|-------------------|----------------|
| Skills | 60.9% | Moderate — classifiers disagree on ~39% of items |
| MCP | 66.8% | Moderate — slightly better consensus |
| Plugins | 81.5% | Good — classifiers mostly agree |

- **Plugins** have the highest agreement, likely because plugin code patterns are more distinct (clearly malicious or clearly benign).
- **Skills** have the lowest agreement, which makes sense — skills are prompt-based text, and the boundary between "suspicious prompt" and "creative prompt" is subjective even for ML models.
- For items where all 3 models disagree (rare), the notebook takes the **most conservative** label (highest severity). This means the L3 counts may be slightly inflated, which the authors acknowledge as a deliberate design choice — better to over-flag than under-flag.

### Key Discrepancies Between This Notebook and the Paper

This notebook is a validation exercise. Several results diverge from the paper's claims:

| Finding | Paper Claims | Notebook Computes | Likely Cause |
|---------|-------------|-------------------|--------------|
| Skills L3 count | 1,637 | 2,277 (+640) | Paper may deduplicate slugs or use different consensus rules |
| MCP L3 count | 164 | 241 (+77) | 167 duplicate slugs inflate consensus set |
| Heuristic any-flag rate | 9.47% | 8.01% (−1.46pp) | Heuristic CSV has 93,735 rows vs 97,544 in distilled set — ~3,800 items missing from heuristic run |
| Heuristic Defender recall | 100% | 7.84% (4/51) | Paper likely refers to a different heuristic configuration or threshold |

These discrepancies don't invalidate the paper's core conclusions — the directional findings all replicate (plugins riskiest, trust signals unreliable, signature scanners miss most threats). But they highlight the importance of version-pinning data pipelines and documenting exact preprocessing steps.

### Summary

The big picture from this validation:

1. **~2–5% of extensions in the Claude ecosystem are classified as high-risk (L3)**, depending on substrate. Plugins are the riskiest channel.
2. **Existing signature-based security scanners catch less than 3% of these threats.** ML-based classification finds 30–50× more dangerous items than traditional tools.
3. **Stars, installs, and badges are not safety signals.** Users cannot rely on social proof to avoid dangerous extensions.
4. **The three substrates have statistically distinct risk profiles** — a one-size-fits-all security approach won't work.
5. **The classifier pipeline is reproducible** — local and remote data agree at >99.4% for all substrates.
6. **Some paper claims don't exactly replicate**, primarily around L3 counts and heuristic recall. The differences are explainable by deduplication and pipeline versioning, and the directional conclusions hold.